# Gold Layer - Data Modeling & KPI Generation

### 

This notebook builds the Gold layer by creating dimension and fact tables, followed by a curated dataset for business analytics.

Key operations performed:
- Creation of dimension tables (customer, product, seller, date)
- Creation of fact tables for transactional data
- Construction of a curated dataset by joining all relevant tables
- Computation of business KPIs from the curated dataset

This layer provides analytics-ready data for reporting and decision-making.

## Setup

In [0]:
from pyspark.sql.functions import *

spark.sql("CREATE SCHEMA IF NOT EXISTS e_commerce.gold")
spark.sql("USE CATALOG e_commerce")
spark.sql("USE SCHEMA gold")

## Dimension Tables

dim_customer

In [0]:
dim_customer = spark.table("e_commerce.silver.silver_customers") \
    .select("customer_id", "customer_city", "customer_state")

dim_customer.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce.gold.dim_customer")

dim_product

In [0]:
dim_product = spark.table("e_commerce.silver.silver_products") \
    .select("product_id", "product_category_name")

dim_product.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce.gold.dim_product")

dim_seller

In [0]:
dim_seller = spark.table("e_commerce.silver.silver_sellers") \
    .select("seller_id", "seller_city", "seller_state")

dim_seller.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce.gold.dim_seller")

dim_date

In [0]:
orders = spark.table("e_commerce.silver.silver_orders")

dim_date = orders.select(
    col("order_purchase_timestamp").alias("date")
).filter(col("order_purchase_timestamp").isNotNull()) \
.dropDuplicates() \
.withColumn("year", year("date")) \
.withColumn("month", month("date")) \
.withColumn("day", dayofmonth("date"))

dim_date.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce.gold.dim_date")


##  Fact Tables

fact_orders

In [0]:
orders = spark.table("e_commerce.silver.silver_orders")
payments = spark.table("e_commerce.silver.silver_order_payments")

# Aggregate payments first (performance optimization)
payments_agg = payments.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_amount"))

fact_orders = orders.join(payments_agg, "order_id", "left") \
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "delivery_time_days",
        "is_late_delivery",
        "is_delivered",
        "total_amount"
    )

fact_orders.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("order_status") \
    .saveAsTable("e_commerce.gold.fact_orders")

fact_order_items

In [0]:
fact_order_items = spark.table("e_commerce.silver.silver_order_items") \
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    )

fact_order_items.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce.gold.fact_order_items")

fact_payments

In [0]:
fact_payments = spark.table("e_commerce.silver.silver_order_payments") \
    .select(
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value"
    )

fact_payments.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce.gold.fact_payments")

fact_reviews

In [0]:
fact_reviews = spark.table("e_commerce.silver.silver_reviews") \
    .select(
        "review_id",
        "order_id",
        "review_score",
        "is_low_rating"
    )

fact_reviews.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce.gold.fact_reviews")

In [0]:
spark.catalog.listTables("e_commerce.gold")

In [0]:
# STAR SCHEMA 


#           dim_customer
#                |
#           fact_orders
#                |
# fact_order_items —— dim_product
#         |
#     dim_seller

# fact_payments
# fact_reviews

## Curated Dataset

In [0]:
from pyspark.sql.functions import *


orders = spark.table("e_commerce.gold.fact_orders")
order_items = spark.table("e_commerce.gold.fact_order_items")
payments = spark.table("e_commerce.gold.fact_payments")
reviews = spark.table("e_commerce.gold.fact_reviews")

customers = spark.table("e_commerce.gold.dim_customer")
products = spark.table("e_commerce.gold.dim_product")
sellers = spark.table("e_commerce.gold.dim_seller")

payments_agg = payments.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_payment"))
reviews_dedup = reviews.groupBy("order_id") \
    .agg(
        avg("review_score").alias("review_score"),
        max("is_low_rating").alias("is_low_rating")
    )


curated = order_items \
    .join(orders, "order_id", "left") \
    .join(customers, "customer_id", "left") \
    .join(products, "product_id", "left") \
    .join(sellers, "seller_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews_dedup, "order_id", "left")

curated.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.gold.gold_curated_orders")

display(curated.limit(10))

In [0]:
# curated.groupBy("order_id", "order_item_id") \
#     .count().filter("count > 1").show()